# Taller 10
**Andres Serrano y Luis Robles**

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

def entrenar(activacion, X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test, semilla=42):
    modelo = Pipeline([
        ("escala", StandardScaler()),
        ("mlp", MLPClassifier(hidden_layer_sizes=(64,), activation=activacion,
                              solver="lbfgs", max_iter=1000, random_state=semilla)),
    ])
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    segundos = time.perf_counter() - inicio
    return {
        "Activación": activacion,
        "Accuracy train": accuracy_score(y_train, modelo.predict(X_train)),
        "Accuracy test": accuracy_score(y_test, modelo.predict(X_test)),
        "Iteraciones": modelo.named_steps["mlp"].n_iter_,
        "Segundos": segundos,
    }

print(X.shape, np.unique(y))

(1797, 64) [0 1 2 3 4 5 6 7 8 9]


## Identity
$f(x) = x$. No introduce no linealidad: la red se comporta como un modelo lineal.

In [ ]:
r_identity = entrenar("identity")
pd.DataFrame([r_identity]).round(4)

,Activación,Accuracy train,Accuracy test,Iteraciones,Segundos
0,identity,1.0,0.9444,18,2.6916


## Logistic
$f(x) = \frac{1}{1 + e^{-x}}$. Salida en $(0, 1)$; satura en los extremos y el gradiente se desvanece.

In [ ]:
r_logistic = entrenar("logistic")
pd.DataFrame([r_logistic]).round(4)

## Tanh
$f(x) = \tanh(x)$. Salida en $(-1, 1)$ y centrada en cero; también satura en los extremos.

In [ ]:
r_tanh = entrenar("tanh")
pd.DataFrame([r_tanh]).round(4)

## ReLU
$f(x) = \max(0, x)$. Es simple de calcular y no satura para $x > 0$.

In [ ]:
r_relu = entrenar("relu")
pd.DataFrame([r_relu]).round(4)

## Comparación

In [ ]:
resumen = pd.DataFrame([r_identity, r_logistic, r_tanh, r_relu]).set_index("Activación")
resumen.round(4)

## Repetición con 10 particiones

In [7]:
activaciones = ["identity", "logistic", "tanh", "relu"]
filas = []
for semilla in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=semilla)
    for a in activaciones:
        r = entrenar(a, Xtr, Xte, ytr, yte, semilla)
        filas.append({"Activación": a, "Accuracy test": r["Accuracy test"], "Segundos": r["Segundos"]})

repeticion = pd.DataFrame(filas).groupby("Activación").agg(
    accuracy_media=("Accuracy test", "mean"),
    accuracy_desv=("Accuracy test", "std"),
    segundos_medio=("Segundos", "mean"),
).loc[activaciones]
repeticion.round(4)

,accuracy_media,accuracy_desv,segundos_medio
Activación,,,
identity,0.9542,0.0093,0.7006
logistic,0.9644,0.0070,0.7719
tanh,0.9694,0.0060,0.7462
relu,0.9689,0.0060,0.6696


## Conclusión
En la partición principal, **relu** obtuvo la mayor accuracy en prueba (0.9806) y **tanh** fue la más rápida, aunque todas entrenan en centésimas de segundo. Al repetir con 10 particiones, relu y tanh quedan prácticamente empatadas (≈0.969), por encima de logistic (0.964) e identity (0.954), que es la menos precisa por ser lineal. Logistic fue la más lenta. Elegimos **"relu"**: fue la mejor en precisión, tiempo de entrenamiento bajo y es la activación estándar para este tipo de redes.